# Multicoordinate Wave Project

Generate one wave project containing several coordinate systems and inspect the coordinate-specific artifacts.

Navigation: [Index](../index.ipynb) | Previous: [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb) | Next: [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)

## Why This Matters
Some workflows compare coordinate systems in one generated executable, so each coordinate system receives its own generated kernels and diagnostics.

## What You Will See
You will see coordinate-specific directories, generated parameters, build evidence, run evidence, and complete diagnostic rows.

Prerequisite bridge: this notebook follows [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb). Terms are defined here before they are used.

## Generate and Run the Multicoordinate Project
The project contains coordinate-specific directories and diagnostics for Cartesian, Spherical, SinhCartesian, and SinhSpherical grids.

In [1]:
from pathlib import Path
import re
import subprocess
import sys
import tempfile

PROJECT_NAME = "wave_equation_multicoordinates"
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_tutorial_multi_", dir=Path.cwd())
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

command = [sys.executable, "-m", "nrpy.examples.wave_equation_multicoordinates"]
print("generator command: python -m nrpy.examples.wave_equation_multicoordinates")
generate = subprocess.run(
    command,
    cwd=WORKSPACE,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=300,
)
print(generate.stdout.replace(str(WORKSPACE), "<workspace>"))
assert PROJECT_DIR.is_dir()

parfile = PROJECT_DIR / "wave_equation_multicoordinates.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.2")
par_text = par_text.replace("diagnostics_output_every = 0.2", "diagnostics_output_every = 0.1")
par_text = par_text.replace("output_progress_every = 1", "output_progress_every = 1000000")
parfile.write_text(par_text, encoding="utf-8")
print(f"--- runtime {parfile.name} ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))

coordinate_dirs = [path.name for path in PROJECT_DIR.iterdir() if path.is_dir() and path.name not in {"MoL", "diagnostics", "intrinsics"}]
print("coordinate directories:")
for name in sorted(coordinate_dirs):
    print(name)
assert {"Cartesian", "SinhCartesian", "SinhSpherical", "Spherical"}.issubset(set(coordinate_dirs))

print("complete project inventory:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

print("\n--- wave_equation_multicoordinates.par ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))

build = subprocess.run(
    ["make", "-j2"],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=300,
)
print("build completed")
print("compiler output line count:", len(build.stdout.splitlines()))

run = subprocess.run(
    [f"./{PROJECT_NAME}", "2.0"],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=120,
)
cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", run.stdout)
print("run output:")
for line in cleaned.replace(str(WORKSPACE), "<workspace>").splitlines():
    if line.strip():
        print(line.rstrip())

print("diagnostics:")
diagnostics = sorted(PROJECT_DIR.glob("out0d-grid*.txt"))
assert diagnostics
for diagnostic in diagnostics:
    print(f"\n--- {diagnostic.name} ---")
    text = diagnostic.read_text(encoding="utf-8", errors="replace")
    print(text)
    assert len(text.strip().splitlines()) >= 2

generator command: python -m nrpy.examples.wave_equation_multicoordinates


In 0.008s, worker completed task 'register_CFunction_initial_data'
In 0.003s, worker completed task 'register_CFunction_diagnostics_nearest'
In 0.034s, worker completed task 'register_CFunction_diagnostics_nearest_2d_xy_and_yz_planes'
In 0.034s, worker completed task 'register_CFunction_diagnostics_nearest_2d_xy_and_yz_planes'
In 0.038s, worker completed task 'register_CFunction_diagnostics_nearest_grid_center'
In 0.039s, worker completed task 'register_CFunction_diagnostics_nearest_grid_center'
In 0.041s, worker completed task 'register_CFunction_diagnostics_nearest_grid_center'
In 0.042s, worker completed task 'register_CFunction_diagnostics_nearest_2d_xy_and_yz_planes'
In 0.041s, worker completed task 'register_CFunction_diagnostics_nearest_grid_center'
In 0.044s, worker completed task '_register_CFunction_diagnostics'
In 0.044s, worker completed task 'register_CFunction_diagnostics_nearest_1d_y_and_z_axes'
In 0.045s, worker completed task 'register_CFunction_diagnostics_nearest_1d_

build completed
compiler output line count: 82


run output:
It: 0 t=0.000 / 0.2 = 0.00% dt=1/483.7 | t/h=0.00 ETA 0h00m00s
WRITING CHECKPOINT: cd struct size = 168 time=0.000000e+00
FINISHED WRITING CHECKPOINT
diagnostics:

--- out0d-grid00-Cartesian-conv_factor-2.00.txt ---
# time DIAG_RELERROR_UU(2) DIAG_RELERROR_VV(3) DIAG_UNUM(4) DIAG_UEXACT(5)
0.000000000000000e+00 0.000000000000000e+00 0.000000000000000e+00 3.997966529243731e+00 3.997966529243731e+00
9.923527388139200e-02 1.001377001799009e-09 -1.219801401297098e-06 3.994691040667144e+00 3.994691036666953e+00


--- out0d-grid01-SinhCartesian-conv_factor-2.00.txt ---
# time DIAG_RELERROR_UU(2) DIAG_RELERROR_VV(3) DIAG_UNUM(4) DIAG_UEXACT(5)
0.000000000000000e+00 0.000000000000000e+00 0.000000000000000e+00 3.999990757843875e+00 3.999990757843875e+00
9.923527388139200e-02 4.561369981281592e-09 -4.705439066190487e-06 3.996709750973538e+00 3.996709732743066e+00


--- out0d-grid02-SinhSpherical-conv_factor-2.00.txt ---
# time DIAG_RELERROR_UU(2) DIAG_RELERROR_VV(3) DIAG_UNUM(4) DIAG

## Interpreting the Output
The coordinate directories and runtime parameter file show one generated project carrying several coordinate systems. The diagnostics demonstrate that each grid writes its own output using the same executable.

## Where This Leads
- [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb)
- [Curvilinear Boundary Conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)